# JA·LE — Colab Training & Evaluation

**TVS Credit E.P.I.C 8.0 · Problem (e) Swarm Intelligence Lending Network**

This notebook is the full-quality version of the JA·LE pipeline. The sandbox V1 (scipy + scikit-learn, 18 s, verified) is reproduced here as a **sanity baseline**, then extended with the parts that could not run there:

1. **PyTorch Geometric GNNs** — GraphSAGE, GAT, GCN, CARE-GNN (simplified), BWGNN
2. **Real public benchmarks** — YelpChi, Amazon, T-Finance, DGraph-Fin
3. **Scale** — the FULL profile (120,000 persons)

---
### What is verified and what is not — read this first

| Component | Status |
|---|---|
| Synthetic generator, entity resolution, graph + feature builder, sklearn baselines, all leakage audits | **Executed and verified** in the sandbox. Numbers in `reports/v1_SMOKE.json`. |
| `jale/models/torch_gnn.py` | **Never executed.** No torch in the sandbox. Syntax-checked only. Expect shape errors on the first run — that is a known gap, not a surprise. |
| `jale/data/public_datasets.py` | **Never executed.** Same reason. Dataset schemas were written from the papers and dataset cards, not from inspecting the files. |

If a cell fails, fix it and record what you changed — the point of this notebook is to produce trustworthy numbers, not to look like it ran.


## 0. Environment


In [ ]:
# Runtime -> Change runtime type -> GPU (T4) for the GNN section.
# CPU is fine for sections 1-4 and for YelpChi.
import subprocess, sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'torch_geometric', 'pyarrow', 'dgl'])   # dgl: real fraud benchmarks

import torch
print('torch        :', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU           :', torch.cuda.get_device_name(0))
import torch_geometric
print('PyG           :', torch_geometric.__version__)


## 1. Get the code

Two options. **Option A** (upload) always works. **Option B** (Drive) is better if you are iterating.

**Option A:** zip the `jale/` project directory on your machine, then `Files -> Upload to session storage`, and run the cell below.


In [ ]:
import os, zipfile

if os.path.exists('jale.zip'):
    with zipfile.ZipFile('jale.zip') as z: z.extractall('.')
    print('extracted jale.zip')
else:
    # Option B: Google Drive
    from google.colab import drive
    drive.mount('/content/drive')
    PROJ = '/content/drive/MyDrive/jale'      # <-- edit to your path
    os.chdir(PROJ)
    print('using', PROJ)

sys_path_added = os.getcwd()
import sys
if sys_path_added not in sys.path: sys.path.insert(0, sys_path_added)

from jale.config import SMOKE, FULL
print('import OK')


## 2. Configuration


In [ ]:
from jale.config import ObservationTime

PROFILE   = SMOKE          # or FULL for 120k persons (~minutes, more RAM)
OBS_TIME  = ObservationTime.APPLICATION   # no repayment history: the honest setting
N_SPLITS  = 5
SEED      = 0
DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'profile={PROFILE.name if hasattr(PROFILE,"name") else PROFILE}  obs={OBS_TIME}  device={DEVICE}')


## 3. Build the synthetic portfolio, graph and features

`build_dataset` writes observables to `raw/` and ground truth to `labels/`. Everything downstream reads **only** `raw/`. The assertion below is the leakage defence: if any ground-truth column ever reaches the feature tables, this fails.


In [ ]:
import numpy as np, pandas as pd
from pathlib import Path
from jale.data.generator import build as build_dataset
from jale.graph.builder import build_graph, fold_groups
from jale.features.builder import build_node_features, build_graph_features

root = build_dataset(PROFILE, 'data/jale_colab')
tabs = {f.stem: pd.read_parquet(f) for f in sorted((root/'raw').glob('*.parquet'))}

BANNED = ('ring_id', 'human_id', 'is_kiosk')
for name, df in tabs.items():
    bad = [c for c in df.columns if c.startswith('label') or c in BANNED]
    assert not bad, f'{name} leaks ground truth: {bad}'
print('label-segregation assertion: PASS')

apps = tabs['applications']
lab  = pd.read_parquet(root/'labels'/'application_labels.parquet')
y = (lab.set_index('application_id')['ring_id']
        .reindex(apps['application_id']).fillna(0).to_numpy() > 0).astype(int)
print(f'{len(apps):,} applications | {y.sum():,} fraud ({y.mean():.2%})')


In [ ]:
graph = build_graph(apps, tabs['guarantor_links'], tabs['persons'])
A = graph.cooccurrence_union()
groups = fold_groups(graph).to_numpy()
print('relations:', {r: graph.incidence[r].shape[1] for r in graph.relations()})
print(f'graph: {A.shape[0]:,} nodes, {A.nnz:,} directed edges, avg degree {A.nnz/A.shape[0]:.1f}')
print(f'ring-disjoint fold groups: {len(np.unique(groups)):,} (largest {pd.Series(groups).value_counts().max()})')


In [ ]:
nf = build_node_features(apps, tabs['emi_schedule'], OBS_TIME)
gf = build_graph_features(graph, apps, nf)

NODE_COLS = [c for c in nf.columns if not c.endswith(('_freq', '_code'))]
G_COLS    = [c for c in gf.columns if c != 'ppr']
X_node = nf.reindex(graph.app_ids)[NODE_COLS]
X_all  = X_node.join(gf.reindex(graph.app_ids)[G_COLS], how='left')

# Observation-time gate: a repayment-history column must never appear here.
HIST = ('n_missed', 'dpd', 'miss_rate', 'ever_dpd30', 'first_missed')
leaked = [c for c in X_all.columns if any(h in c for h in HIST)]
assert not leaked, f'repayment history leaked: {leaked}'

print(f'node features: {len(NODE_COLS)} | graph features: {len(G_COLS)} | total: {X_all.shape[1]}')
print('observation-time gate: PASS')


## 4. sklearn baselines under ring-disjoint CV

This reproduces the sandbox numbers. On SMOKE, with the **pinned** environment in `requirements.txt` (numpy 2.5 / pandas 3.0 / scikit-learn 1.8), the expected output is:

| Model | AUC-PR | Lift |
|---|---|---|
| best single node feature (`n_guarantors`), raw | 0.233 | 6.2x |
| node-only logistic | 0.125 | 3.3x |
| node-only GBT | 0.242 | 6.4x |
| graph-only GBT | 0.732 | 19.4x |
| **node + graph GBT** | **0.729** | **19.3x** |
| node + graph GBT, nested CV (the honest headline) | 0.700 | 18.6x |
| graph-regularised LR | 0.407 | 10.8x |

> The `HistGradientBoosting` fits shift by ~0.02–0.03 AUC-PR across scikit-learn
> versions — this is why `requirements.txt` is pinned and `reports/v1_SMOKE.json`
> carries a regeneration date. An unpinned install will not match exactly; the
> *shape* of the result (graph >> node, ring-disjoint << random, control at
> chance) is what is stable.
>
> **Read the first row.** `n_guarantors` alone gives 6.2x lift. An earlier version
> reported node-only GBT at 0.038 (chance) because of a `min_samples_leaf=20` bug
> — with ~26 positives per fold a leaf needing 20 samples cannot split on the
> positive class. `best_single_feature()` runs every time now so a broken baseline
> cannot flatter the model again.


In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from jale.models.models import fit_logistic, fit_gbt, GraphRegularisedLogistic, GBT_DEFAULTS
from jale.eval.metrics import full_metrics, best_single_feature

print('GBT defaults:', GBT_DEFAULTS)   # leaf=10, not 20 -- see the note above

def prep(X, fit_mask):
    Xv = X.replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(dtype=float)
    return StandardScaler().fit(Xv[fit_mask]).transform(Xv)

def cv(X, y, groups, kind):
    oof = np.zeros(len(y))
    for tr, te in GroupKFold(n_splits=N_SPLITS).split(X, y, groups=groups):
        m = np.zeros(len(y), dtype=bool); m[tr] = True
        Xtr, Xte = prep(X, m)[tr], prep(X, m)[te]
        if kind == 'lr':      s = fit_logistic(Xtr, y[tr]).decision_function(Xte)
        elif kind == 'gbt':   s = fit_gbt(Xtr, y[tr], seed=SEED).predict_proba(Xte)[:, 1]
        elif kind == 'glr':   s = GraphRegularisedLogistic(lam=0.5).fit(Xtr, y[tr], A[tr][:, tr]).decision_function(Xte)
        oof[te] = s
    return oof

print('\n--- honest floor: best single column used as a raw score ---')
for lbl, X in [('node', X_node), ('graph', gf.reindex(graph.app_ids)[G_COLS])]:
    nm, auc = best_single_feature(y, X)
    print(f'  best single {lbl:5s} feature: {nm:32s} AUC-PR={auc:.4f} lift={auc/y.mean():.1f}x')

print('\n--- models ---')
base = {}
for name, X, kind in [('node-only LR', X_node, 'lr'), ('node-only GBT', X_node, 'gbt'),
                      ('graph-only GBT', gf.reindex(graph.app_ids)[G_COLS], 'gbt'),
                      ('node+graph GBT', X_all, 'gbt'), ('node+graph graph-LR', X_all, 'glr')]:
    m = full_metrics(y, cv(X, y, groups, kind)); base[name] = m
    print(f'{name:20s} AUC-PR={m["auc_pr"]:.4f} lift={m["lift_pr"]:.1f}x AUC-ROC={m["auc_roc"]:.4f} R@5%={m["recall_at_5pct"]:.3f}')


## 5. PyTorch GNNs on the SAME ring-disjoint folds

This is the comparison that matters. Same graph, same features, same folds, same metric — only the model class changes. Anything else is not a fair comparison.

> **Unverified code.** `jale/models/torch_gnn.py` has never been executed. If a > shape error appears, it is a bug in that file, not in your setup.


In [ ]:
from jale.models.torch_gnn import to_pyg_data, build_model, train_and_eval
from jale.graph.builder import STRONG_FOLD_RELATIONS

# TWO fixes here, both of which matter:
#
# 1. The scaler is fitted INSIDE each fold on training rows only. Fitting it once
#    on the whole matrix leaks each test fold's mean and variance into training.
#
# 2. Message passing uses the STRONG relations only (device/account/person/
#    guarantor), NOT the full cooccurrence_union(). Measured: cooccurrence_union()
#    has 511,652 edges of which 403,062 (78.8%) cross fold boundaries, and 98.6% of
#    them come from the dealer relation. A transductive GNN would pass messages
#    from train nodes into test nodes along those edges, destroying the ring-
#    disjoint guarantee. The strong-relations graph has 8,294 edges and ZERO
#    cross-fold edges, so folds are genuinely disconnected components.
#
# Dealer stays in as FEATURES (worth 0.098 AUC-PR, measured) but not as EDGES: at
# 98.6% of all edges it turns the graph into a dense dealer blob that carries no
# fraud structure, which is a large part of why GNNs underperformed GBT.
A_msg = graph.cooccurrence_union(tuple(STRONG_FOLD_RELATIONS))
print(f'message-passing graph: {A_msg.nnz:,} edges (vs {A.nnz:,} in the full union)')

Xraw = X_all.replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(dtype=np.float32)
y64 = y.astype(np.int64)

results_gnn = {}
fold_iter = list(GroupKFold(n_splits=N_SPLITS).split(Xraw, y64, groups=groups))

for kind in ['sage', 'gat', 'gcn', 'caregnn', 'bwgnn']:
    per_fold = []
    for fi, (tr, te) in enumerate(fold_iter):
        sc = StandardScaler().fit(Xraw[tr])          # train rows only
        Xv = sc.transform(Xraw).astype(np.float32)   # transform all: the GNN needs
        va = te[: len(te) // 2]; te2 = te[len(te) // 2:]   # features for every node
        trm = np.zeros(len(y), bool); trm[tr] = True
        vam = np.zeros(len(y), bool); vam[va] = True
        tem = np.zeros(len(y), bool); tem[te2] = True
        data = to_pyg_data(A_msg, Xv, y64, trm, vam, tem)
        model = build_model(kind, in_dim=Xv.shape[1], hidden=128)
        _, met = train_and_eval(model, data, epochs=150, lr=1e-2, device=DEVICE)
        per_fold.append(met)
    agg = {k: float(np.mean([m[k] for m in per_fold])) for k in per_fold[0]}
    results_gnn[kind] = agg
    print(f'{kind:9s} AUC-PR={agg["auc_pr"]:.4f} AUC-ROC={agg["auc_roc"]:.4f}')


## 6. Leakage audits — run these no matter what

None of the numbers above mean anything unless these pass. A GNN that scores well on the shuffled-label control is memorising, not detecting.


In [ ]:
from jale.eval.splits import shuffle_label_control

fold_id = np.zeros(len(y), dtype=int)
for i, (_, te) in enumerate(fold_iter): fold_id[te] = i

yl = shuffle_label_control(y, fold_id, seed=1)
assert float(np.mean(yl)) == float(np.mean(y)), 'control changed the base rate'
print(f'shuffled-label control: base rate preserved at {yl.mean():.4f}')

for name, X in [('node-only GBT', X_node), ('node+graph GBT', X_all)]:
    m = full_metrics(yl, cv(X, yl, groups, 'gbt'))
    ok = m['auc_pr'] < 1.5 * m['base_rate']
    print(f'  {name:16s} AUC-PR={m["auc_pr"]:.4f} vs base {m["base_rate"]:.4f} -> {"PASS" if ok else "FAIL: LEAKAGE"}')


In [ ]:
# Random split vs ring-disjoint: quantifies how much a ring leaks into a naive eval.
from sklearn.model_selection import StratifiedKFold
oof = np.zeros(len(y))
for tr, te in StratifiedKFold(N_SPLITS, shuffle=True, random_state=SEED).split(X_all, y):
    m = np.zeros(len(y), bool); m[tr] = True
    Xtr, Xte = prep(X_all, m)[tr], prep(X_all, m)[te]
    oof[te] = fit_gbt(Xtr, y[tr], seed=SEED).predict_proba(Xte)[:, 1]
leaky = full_metrics(y, oof)
print(f'random split      AUC-PR = {leaky["auc_pr"]:.4f}')
print(f'ring-disjoint     AUC-PR = {base["node+graph GBT"]["auc_pr"]:.4f}')
print(f'leakage gap              = {leaky["auc_pr"] - base["node+graph GBT"]["auc_pr"]:+.4f}')


## 6b. Nested CV — the number that goes in the report

Choosing hyperparameters on the same folds used for reporting is selection on the
test set. On SMOKE it is worth **+0.037 AUC-PR**. This cell removes it: selection
happens inside each outer training split only, so the outer test fold is never seen.

Slow (~11 min on SMOKE, longer on FULL). **This is the headline number.**


In [ ]:
from jale.models.models import select_gbt

nested = {}
KEYS = {'node-only': 'node-only GBT', 'graph-only': 'graph-only GBT',
        'node+graph': 'node+graph GBT'}
for lbl, X in [('node-only', X_node), ('graph-only', gf.reindex(graph.app_ids)[G_COLS]),
               ('node+graph', X_all)]:
    Xv = X.replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(float)
    oof = np.zeros(len(y))
    for tr, te in GroupKFold(n_splits=N_SPLITS).split(X, y, groups=groups):
        params, _ = select_gbt(Xv[tr], y[tr], groups[tr])
        sc = StandardScaler().fit(Xv[tr])
        m = fit_gbt(sc.transform(Xv[tr]), y[tr], seed=SEED,
                    max_depth=params[0], min_samples_leaf=params[1],
                    max_iter=params[2], learning_rate=params[3])
        oof[te] = m.predict_proba(sc.transform(Xv[te]))[:, 1]
    r = full_metrics(y, oof); nested[lbl] = r
    print(f'{lbl:11s} nested AUC-PR={r["auc_pr"]:.4f} lift={r["lift_pr"]:.1f}x '
          f'R@5%={r["recall_at_5pct"]:.3f}  (fixed-param: {base[KEYS[lbl]]["auc_pr"]:.4f})')


## 7. Real public benchmarks

**Start with YelpChi.** It is small, downloads automatically, and proves the harness works. T-Finance and DGraph-Fin are where the interesting results are, and where the runtime is most likely to die.

We apply **our** ring-disjoint protocol to **their** data. Published numbers on these datasets use random splits, so our numbers will be lower — that is the point, and it must be stated wherever the comparison appears.


In [ ]:
from jale.data.public_datasets import (load_yelpchi, load_amazon, load_bwgnn_pt,
                                       load_dgraphfin, subsample,
                                       ring_disjoint_folds_from_adjacency)

g = load_yelpchi()          # ~200 MB, fits free Colab
print(g.summary())
g = subsample(g, 0.25)      # drop to 25% if memory is tight
print(g.summary())
gfolds = ring_disjoint_folds_from_adjacency(g.A, n_splits=N_SPLITS)
print('fold sizes:', np.bincount(gfolds))


In [ ]:
# Same ring-disjoint CV, now on real data. sklearn first (fast, always works).
Xraw_r = np.nan_to_num(g.X, nan=0.0, posinf=0.0, neginf=0.0)
yr = g.y.astype(int)
oof = np.zeros(len(yr))
for tr, te in GroupKFold(n_splits=N_SPLITS).split(Xraw_r, yr, groups=gfolds):
    sc = StandardScaler().fit(Xraw_r[tr])      # fitted on training rows only
    Xtr, Xte = sc.transform(Xraw_r[tr]), sc.transform(Xraw_r[te])
    oof[te] = fit_gbt(Xtr, yr[tr], seed=SEED).predict_proba(Xte)[:, 1]
m = full_metrics(yr, oof)
print(f'{g.name} ring-disjoint  AUC-PR={m["auc_pr"]:.4f} AUC-ROC={m["auc_roc"]:.4f} R@5%={m["recall_at_5pct"]:.3f}')

# Compare against the SAME model on a random split of the same data.
oof2 = np.zeros(len(yr))
for tr, te in StratifiedKFold(N_SPLITS, shuffle=True, random_state=SEED).split(Xraw_r, yr):
    sc = StandardScaler().fit(Xraw_r[tr])
    oof2[te] = fit_gbt(sc.transform(Xraw_r[tr]), yr[tr], seed=SEED).predict_proba(
        sc.transform(Xraw_r[te]))[:, 1]
m2 = full_metrics(yr, oof2)
print(f'{g.name} random split    AUC-PR={m2["auc_pr"]:.4f} AUC-ROC={m2["auc_roc"]:.4f}')
print(f'gap on REAL data          = {m2["auc_pr"] - m["auc_pr"]:+.4f}')


In [ ]:
# GNN on real data -- only after section 5 runs clean.
data = to_pyg_data(g.A, Xraw_r.astype(np.float32), yr.astype(np.int64),
                   gfolds != 0, gfolds == 1, gfolds == 2)
for kind in ['sage', 'caregnn', 'bwgnn']:
    model = build_model(kind, in_dim=Xraw_r.shape[1], hidden=128)
    _, met = train_and_eval(model, data, epochs=200, device=DEVICE)
    print(f'{g.name} {kind:9s} AUC-PR={met["auc_pr"]:.4f} AUC-ROC={met["auc_roc"]:.4f}')


In [ ]:
# Larger benchmarks -- uncomment one at a time. GPU runtime required.
# g = load_bwgnn_pt('/content/data/T-Finance.pt', 'T-Finance'); print(g.summary())
# g = load_dgraphfin('/content/data/dgraphfin.npz');           print(g.summary())
# g = subsample(g, 0.10)   # DGraph-Fin has 3.7M nodes; subsample hard


## 8. Ablations — the experiments that actually convince a reviewer

Run these. They are more persuasive than any single headline number.


In [ ]:
# (a) Per-relation ablation: which relations carry the signal?
for rel in graph.relations():
    keep = [c for c in G_COLS if f'_{rel}' not in c and not c.startswith(f'{rel}_')]
    Xa = X_node.join(gf.reindex(graph.app_ids)[keep], how='left')
    m = full_metrics(y, cv(Xa, y, groups, 'gbt'))
    print(f'drop {rel:10s} -> AUC-PR={m["auc_pr"]:.4f}  (full={base["node+graph GBT"]["auc_pr"]:.4f})')


In [ ]:
# (b) Train on seen typologies, test on an unseen one.
#     Does the model learn RINGS, or does it learn THESE rings?
rings = pd.read_parquet(root/'labels'/'rings.parquet')
appl  = pd.read_parquet(root/'labels'/'application_labels.parquet')
typ   = appl.merge(rings[['ring_id','typology']], on='ring_id', how='left')
typ   = typ.set_index('application_id')['typology'].reindex(graph.app_ids)

for held in sorted(typ.dropna().unique()):
    te = (typ == held).to_numpy()
    tr = ~te
    Xtr, Xte = prep(X_all, tr)[tr], prep(X_all, tr)[te]
    s = fit_gbt(Xtr, y[tr], seed=SEED).predict_proba(Xte)[:, 1]
    mm = full_metrics(y[te], s)
    print(f'held out {held:18s} n={te.sum():4d} pos={int(y[te].sum()):3d} '
          f'AUC-PR={mm.get("auc_pr", float("nan")):.4f}')


## 9. Summary

Copy this table into the report. Report the **ring-disjoint** numbers as the headline. The random-split numbers exist only to quantify the leakage.


In [ ]:
rows = [('sklearn ' + k, v['auc_pr'], v['auc_roc']) for k, v in base.items()]
rows += [('GNN ' + k, v['auc_pr'], v['auc_roc']) for k, v in results_gnn.items()]
summary = pd.DataFrame(rows, columns=['model', 'auc_pr', 'auc_roc']).sort_values('auc_pr', ascending=False)
display(summary)
summary.to_csv('reports/colab_summary.csv', index=False)


## Known open items

1. **`torch_gnn.py` first execution.** The sklearn/feature path is verified; the GNN file has known first-run risks. CARE-GNN's `_gated_agg` and BWGNN's Chebyshev recursion were fixed once already (see `doubts.md` D7) — re-check tensor shapes and that BWGNN's band-pass response matches the authors' `BetaWavelet.get_filter`.
2. **GNN vs GBT is still open on this graph.** Earlier GNN numbers were run on a leaky, dealer-dominated graph and are invalid (`doubts.md` D9). Re-run with `A_msg` (strong relations only) and the fixed model code before quoting anything.
3. **Cross-typology generalisation (~0.28 mean vs ~0.73 in-distribution).** Real and disclosed. The strongest argument for the structural L4 ring score, which carries no learned weights.
4. **Self-generated data is the ceiling.** `collab-help/05_semisynthetic_real_graph.py` injects known rings into a real graph (real benign background, real density) — the closest external check available.

Implemented since this notebook was written (see `jale/demo/`): **L4 ring-level scoring**, **L5 explanations**, **cold-start propagation**, and **entity resolution in the evaluation path** (`experiments/er_in_path.py` — measured cost ≈ −0.08 AUC-PR). The public loaders now use DGL's `FraudDataset` (the real YelpChi / Amazon-fraud benchmarks), not the co-purchase / GraphSAINT datasets a previous version loaded by mistake.
